# Part 3: Reverse Mode Automatic Differentiation with PyTorch

PyTorch implements dynamic reverse-mode automatic differentiation, much like we did in the previous exercise. There is one major difference between what PyTorch provides and our simple example: it works directly with matrices (`Tensor`s) rather than only with scalars (although a tensor can, of course, represent a scalar).

A tensor tracks the operations used to compute it when `requires_grad=True`. PyTorch can then traverse that computation graph in reverse to accumulate gradients.

We'll start with the simple example we tried earlier in the code block below:

__Task:__ Run the following code and verify the solution is correct.

In [ ]:
import torch

# set up the problem
x = torch.tensor(0.5, requires_grad=True)
y = torch.tensor(4.2, requires_grad=True)
z = x * y + torch.sin(x)

print("z = " + str(z.item()))

z.backward() # this goes through the computation graph and accumulates the gradients in the cached .grad attributes
print("dz/dx = " + str(x.grad.item()))
print("dz/dy = " + str(y.grad.item()))

As with our own AD implementation, PyTorch lets us differentiate through an algorithm.

__Task__: Use the block below to compute the gradient $\partial z/\partial x$ of the following pseudocode algorithm and store the result in the `dzdx` variable:

    x = 0.5
    z = 1
    i = 0
    while i<2:
        z = (z + i) * x * x
        i = i + 1

In [ ]:
dzdx = None

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert dzdx


## PyTorch limitations: in-place operations and aliasing

PyTorch will throw an error at runtime if you try to differentiate through an in-place operation on a tensor. 

__Task__: Run the following code to see this in action.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x.tanh()
y.add_(3) # inplace addition
y.backward()

Aliasing is also something that can't be differentiated through and will result in a slightly more cryptic error.

__Task__: Run the following code to see this in action. If you don't understand what this code does add some `print` statements to show the values of `x` and `y` at various points.

In [ ]:
x = torch.tensor([1, 2, 3, 4], requires_grad=True, dtype=torch.float)
y = x[:1]
y.add_(3)
y.backward()

## Dealing with multiple outputs

PyTorch can deal with the case where there are multiple output variables if we can formulate the expression in terms of tensor operations. Consider the example from the presentation for example:

$$\begin{cases}
     z = 2x + \sin x\\
     v = 4x + \cos x
\end{cases}$$

We could formulate this as:

$$
\begin{bmatrix}z \\ v\end{bmatrix} = \begin{bmatrix}2 \\ 4\end{bmatrix} \odot \bar{x} + \begin{bmatrix}1 \\ 0\end{bmatrix} \odot \sin\bar x + \begin{bmatrix}0 \\ 1\end{bmatrix} \odot \cos\bar x
$$

where 

$$
\bar x = \begin{bmatrix}x \\ x\end{bmatrix}
$$

and $\odot$ represents the Hadamard or element-wise product. This is demonstrated using PyTorch in the following code block.

__Task:__ run the code below.

In [ ]:
x = torch.tensor([[1.0],[1.0]], requires_grad=True)

zv = ( torch.tensor([[2.0],[4.0]]) * x +
         torch.tensor([[1.0], [0.0]]) * torch.sin(x) +
         torch.tensor([[0.0], [1.0]]) * torch.cos(x) )
        
zv.backward(torch.tensor([[1.0],[1.0]])) # Note as we have "multiple outputs" we must pass in a tensor of weights of the correct size

print(x.grad)

__Task:__ Use the following box to write down the derivative of the expression for $\begin{bmatrix}z \\ v\end{bmatrix}$ and verify the gradients $\partial z/\partial x$ and $\partial v/\partial x$ are correct for $x=1$.

YOUR ANSWER HERE

## Gradient descent & gradients with respect to a vector
Let's put everything together and using automatically computed gradients to find the minima of a function by taking steps down the gradient from an initial position. Rather than explicitly creating each input variable as a scalar as in the previous examples, we'll use a vector instead (so our gradients will be with respect to each element of the vector).

__Task:__ work through the following example to see how taking gradients with respect to a vector works & how simple gradient descent optimisation can be implemented.

In [ ]:
# This is our starting vector
initial = [[2.0], [1.0], [10.0]]

# This is the function we will optimise (feel free to work out the analytic minimum!)
def function(x):
    return x[0]**2 + x[1]**2 + x[2]**2

x = torch.tensor(initial, requires_grad=True, dtype=torch.float)
for i in range(100):
    # Gradients accumulate by default, so clear the previous gradient.
    x.grad = None

    # Evaluate the function and compute its gradient at the current x.
    J = function(x)
    J.backward()

    # Parameter updates should not themselves become part of the graph.
    with torch.no_grad():
        x -= x.grad * 0.1

    if i % 10 == 0:
        print((x, function(x).item()))


__Task__: Answer the following question in the box below: Why is the update above performed inside a `torch.no_grad()` block? What would happen to the computation graph if PyTorch tracked every optimisation update?

YOUR ANSWER HERE

## Jacobian--vector products

Forward-mode AD propagates a direction through a function. If

$$
\mathbf{f}:\mathbb{R}^n\rightarrow\mathbb{R}^m,
$$

then a Jacobian--vector product (JVP) computes $J_{\mathbf f}(\mathbf x)\mathbf v$ without first constructing the complete Jacobian. The vector $\mathbf v$ has the same shape as the input and says which input-space direction we want to follow.

Consider the function

$$
\mathbf f(\mathbf x)=
\begin{bmatrix}
x_0x_1\\
\sin(x_0)+x_1^2
\end{bmatrix}.
$$

__Task__: Run the code below. Then derive the Jacobian by hand and verify the reported JVP for the supplied point and direction.

In [ ]:
def vector_function(x):
    return torch.stack((
        x[0] * x[1],
        torch.sin(x[0]) + x[1]**2,
    ))

x_point = torch.tensor([1.0, 2.0])
direction = torch.tensor([0.5, -1.0])

value, jvp_value = torch.func.jvp(
    vector_function,
    (x_point,),
    (direction,),
)

print("f(x) =", value)
print("J v  =", jvp_value)


YOUR ANSWER HERE

__Task__: Change the direction to each of the two standard basis vectors, `torch.tensor([1.0, 0.0])` and `torch.tensor([0.0, 1.0])`. How do the two resulting JVPs relate to the columns of the Jacobian?

## Vector--Jacobian products

Reverse-mode AD starts with weights on the outputs and propagates them back to the inputs. These output weights are sometimes called a **cotangent vector**: here it is enough to think of a cotangent as a vector with one weight for each output.

For an output cotangent $\mathbf c$, a vector--Jacobian product (VJP) computes $\mathbf c^\mathsf{T}J_{\mathbf f}(\mathbf x)$, or equivalently returns the column-vector form $J_{\mathbf f}(\mathbf x)^\mathsf{T}\mathbf c$. PyTorch's `torch.func.vjp` returns the function value and a new function that can apply this reverse pass to any suitably shaped cotangent.

__Task__: Run the following example. Use your Jacobian from the previous task to check the result by hand.

In [ ]:
cotangent = torch.tensor([1.0, 0.5])

value, vjp_fn = torch.func.vjp(vector_function, x_point)
(vjp_value,) = vjp_fn(cotangent)

print("f(x)  =", value)
print("J^T c =", vjp_value)


The familiar `.backward()` operation performs the same kind of reverse computation. For a vector output, the argument passed to `.backward()` supplies the output cotangent.

__Task__: Run the comparison below and explain why calling `.backward()` without an argument worked for the earlier scalar objective but would not be sufficient here.

In [ ]:
x_backward = x_point.clone().requires_grad_()
vector_function(x_backward).backward(cotangent)

print("vjp result:       ", vjp_value)
print(".backward result: ", x_backward.grad)
torch.testing.assert_close(x_backward.grad, vjp_value)


YOUR ANSWER HERE

## Gradient checking

Even when analytic or automatic differentiation will be used for training, numerical differentiation is useful for checking an implementation. `torch.autograd.gradcheck` compares the derivatives produced by autograd with finite-difference approximations.

Finite differences involve subtracting nearby function values, so gradient checking should normally use double precision. It is also a test for small examples, not a practical way to train a model.

__Task__: Run the following check for `vector_function`.

In [ ]:
x_check = torch.tensor(
    [1.0, 2.0],
    dtype=torch.double,
    requires_grad=True,
)

gradient_check_passed = torch.autograd.gradcheck(
    vector_function,
    (x_check,),
)
print("gradient check passed:", gradient_check_passed)


A gradient check is most valuable when it can expose a broken gradient path. The function below returns the same values as `vector_function`, but `.detach()` incorrectly removes those values from the computation graph. Adding `0 * x` gives autograd a path to the input, but its analytic gradient along that path is zero.

__Task__: Run the code and explain why the numerical and autograd derivatives disagree. Why does `raise_exception=False` make this example more convenient in a notebook?

In [ ]:
def incorrectly_detached_function(x):
    return vector_function(x).detach() + 0 * x

broken_check_passed = torch.autograd.gradcheck(
    incorrectly_detached_function,
    (x_check,),
    raise_exception=False,
)
print("broken gradient check passed:", broken_check_passed)

assert gradient_check_passed
assert not broken_check_passed


YOUR ANSWER HERE